# E-Commerce Purchase Prediction — Research Pipeline
## XGBoost (Tabular) | GRU (Sequence) | Hybrid (GRU Embeddings + XGBoost)
---
**Dataset:** eCommerce Behavior Data — Oct + Nov 2019 (Kaggle / mkechinov)  
**Goal:** Predict whether a browsing session will end in a purchase.  
**Hardware target:** 24 GB GPU workstation  
**Run order:** Execute cells top-to-bottom. No manual edits required.

### What gets saved to disk
| File | Description |
|---|---|
| `outputs/gru_best_model.keras` | Full GRU model (architecture + best weights) |
| `outputs/gru_final_weights.weights.h5` | GRU weights only (lightweight backup) |
| `outputs/xgboost_champion.pkl` | XGBoost tabular-only champion model |
| `outputs/xgboost_hybrid.pkl` | XGBoost hybrid champion model |
| `outputs/price_scaler.pkl` | MinMaxScaler fitted on log-price (for inference) |
| `outputs/best_features.json` | EFS-selected feature names |
| `outputs/optimal_thresholds.json` | F2-optimal thresholds for all 3 models |
| `outputs/model_comparison.csv` | Final metrics table |
| `outputs/*.png` | All result plots |

## STEP 1 — Install Dependencies

In [ ]:
!pip install opendatasets mlxtend xgboost tensorflow scikit-learn pandas numpy matplotlib seaborn joblib -q

## STEP 2 — Imports + GPU Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc, time, json, os
import joblib
from IPython.display import display

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_curve, auc,
    f1_score, recall_score, roc_auc_score,
    precision_recall_curve
)

# XGBoost
from xgboost import XGBClassifier
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ── GPU CONFIGURATION FOR 24 GB WORKSTATION ──────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Allow memory growth — prevents TF from grabbing all 24 GB upfront
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU(s) configured: {[g.name for g in gpus]}')
    except RuntimeError as e:
        print(f'GPU config error: {e}')
else:
    print('No GPU found — running on CPU')

print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'Pandas version     : {pd.__version__}')

# ── OUTPUT FOLDER ─────────────────────────────────────────────────
os.makedirs('outputs', exist_ok=True)
print('Output folder ready: ./outputs/')

sns.set_theme(style='whitegrid')

## STEP 3 — Download Dataset from Kaggle
> You will be prompted for your Kaggle username and API key once.

In [ ]:
import opendatasets as od
od.download(
    'https://www.kaggle.com/datasets/mkechinov/'
    'ecommerce-behavior-data-from-multi-category-store/data'
)

## STEP 4 — Load, Concatenate & Clean Data
**Changes from original:**
- `product_id` added to `columns_to_keep` (was missing → caused KeyError in groupby)
- `drop_duplicates()` added (was completely absent)
- Zero-price rows removed
- All steps print shape so you can verify progress

In [ ]:
print('=' * 60)
print('LOADING OCT + NOV 2019 DATA')
print('=' * 60)

path_oct = 'ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv'
path_nov = 'ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv'

# product_id is required for unique_products feature in groupby
columns_to_keep = [
    'event_time', 'event_type', 'product_id',
    'category_code', 'brand', 'price',
    'user_id', 'user_session'
]

print('Reading October...')
df_oct = pd.read_csv(path_oct, usecols=columns_to_keep)
print(f'  Oct shape : {df_oct.shape}')

print('Reading November...')
df_nov = pd.read_csv(path_nov, usecols=columns_to_keep)
print(f'  Nov shape : {df_nov.shape}')

print('Concatenating...')
final_df = pd.concat([df_oct, df_nov], ignore_index=True)
del df_oct, df_nov
gc.collect()
print(f'Shape after concat     : {final_df.shape}')

# ── REMOVE DUPLICATES ─────────────────────────────────────────────
# Duplicate = identical row across all columns (can occur at month boundary)
n_before = len(final_df)
final_df = final_df.drop_duplicates()
n_dropped = n_before - len(final_df)
print(f'Duplicates removed     : {n_dropped:,}  ({n_dropped/n_before*100:.2f}%)')
print(f'Shape after dedup      : {final_df.shape}')

# ── DATETIME CONVERSION ───────────────────────────────────────────
final_df['event_time'] = pd.to_datetime(final_df['event_time'])

# ── REMOVE ZERO-PRICE ROWS ────────────────────────────────────────
n_before = len(final_df)
final_df = final_df[final_df['price'] > 0].copy()
print(f'Zero-price rows removed: {n_before - len(final_df):,}')
print(f'Final clean shape      : {final_df.shape}')

print()
print('Event type distribution:')
print(final_df['event_type'].value_counts())
display(final_df.head())

## STEP 5 — Feature Engineering (Session-Level)
Collapses raw event log into one row per session with 11 behavioral features.

In [ ]:
print('=' * 60)
print('FEATURE ENGINEERING PIPELINE')
print('=' * 60)

# ── EVENT FLAGS ───────────────────────────────────────────────────
final_df['is_view']     = (final_df['event_type'] == 'view').astype(np.int8)
final_df['is_cart']     = (final_df['event_type'] == 'cart').astype(np.int8)
final_df['is_purchase'] = (final_df['event_type'] == 'purchase').astype(np.int8)

# Separate price by event type — prevents data leakage into features
final_df['view_price'] = np.where(final_df['event_type'] == 'view', final_df['price'], np.nan)
final_df['cart_price'] = np.where(final_df['event_type'] == 'cart', final_df['price'], np.nan)

# Time of first cart event per session (used for dwell calculation)
first_cart_time = (
    final_df[final_df['is_cart'] == 1]
    .groupby('user_session')['event_time'].min()
)

# ── MASTER GROUPBY ────────────────────────────────────────────────
ml_dataframe = final_df.groupby('user_session').agg(
    total_views       = ('is_view',     'sum'),
    total_carts       = ('is_cart',     'sum'),
    unique_products   = ('product_id',  'nunique'),
    median_view_price = ('view_price',  'median'),
    median_cart_price = ('cart_price',  'median'),
    max_view_price    = ('view_price',  'max'),
    min_view_price    = ('view_price',  'min'),
    start_time        = ('event_time',  'min'),
    end_time          = ('event_time',  'max'),
    target_purchase   = ('is_purchase', 'max')
)

# Fill NaNs (sessions with no cart have no cart price)
ml_dataframe[['median_view_price', 'median_cart_price']] =     ml_dataframe[['median_view_price', 'median_cart_price']].fillna(0)
ml_dataframe[['max_view_price', 'min_view_price']] =     ml_dataframe[['max_view_price', 'min_view_price']].fillna(0)

# Raw duration — compute before capping (needed for accurate bot detection)
ml_dataframe['session_duration_seconds'] = (
    ml_dataframe['end_time'] - ml_dataframe['start_time']
).dt.total_seconds()

# ── BOT REMOVAL ───────────────────────────────────────────────────
# Bots click > 30 pages/minute AND have > 15 total views
print(f'Sessions before bot removal : {ml_dataframe.shape[0]:,}')
raw_speed = ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)
bot_mask  = (raw_speed > 30) & (ml_dataframe['total_views'] > 15)
ml_dataframe = ml_dataframe[~bot_mask]
print(f'Sessions after bot removal  : {ml_dataframe.shape[0]:,}  '
      f'({bot_mask.sum():,} bots dropped)')

# ── CAP ZOMBIE SESSIONS ───────────────────────────────────────────
# Sessions > 1 hour are people who left the tab open, not real browsing
ml_dataframe['session_duration_seconds'] = np.clip(
    ml_dataframe['session_duration_seconds'], 0, 3600
)

# ── ENGINEERED FEATURES ───────────────────────────────────────────
# 1. Browsing velocity (views per minute)
ml_dataframe['views_per_minute'] = (
    ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)
)

# 2. Focus ratio — re-viewing same items = strong purchase intent
#    e.g. 20 total views / 2 unique products = ratio 10 (obsessed)
ml_dataframe['focus_ratio'] = (
    ml_dataframe['total_views'] / ml_dataframe['unique_products'].replace(0, 1)
)

# 3. How quickly they first added to cart (lower = more decisive)
ml_dataframe['first_cart_time_seconds'] = np.clip(
    (first_cart_time - ml_dataframe['start_time']).dt.total_seconds().fillna(0),
    0, 3600
)

# 4. Time spent AFTER adding to cart (high = hesitation / abandonment risk)
ml_dataframe['post_cart_dwell_seconds'] = np.clip(
    np.where(
        ml_dataframe['total_carts'] > 0,
        ml_dataframe['session_duration_seconds'] - ml_dataframe['first_cart_time_seconds'],
        0
    ),
    0, 3600
)

# 5. Price exploration range (narrow = focused budget, wide = window shopping)
ml_dataframe['price_exploration_capped'] = np.clip(
    ml_dataframe['max_view_price'] - ml_dataframe['min_view_price'],
    0, 1000
)

# 6. Cart-to-view ratio (decisiveness — how many views convert to cart actions)
ml_dataframe['cart_view_ratio'] = np.clip(
    ml_dataframe['total_carts'] / ml_dataframe['total_views'].replace(0, 1),
    0, 1.0
)

# ── CLEANUP ───────────────────────────────────────────────────────
ml_dataframe = ml_dataframe.drop(
    columns=['start_time', 'end_time', 'max_view_price', 'min_view_price']
)

print(f'\nFinal ml_dataframe shape: {ml_dataframe.shape}')
print(f'Features: {[c for c in ml_dataframe.columns if c != "target_purchase"]}')
print(f'\nTarget distribution:')
vc = ml_dataframe['target_purchase'].value_counts()
print(f'  Non-buyers (0): {vc[0]:,}  ({vc[0]/len(ml_dataframe)*100:.2f}%)')
print(f'  Buyers     (1): {vc[1]:,}  ({vc[1]/len(ml_dataframe)*100:.2f}%)')
display(ml_dataframe.head())

---
## MODEL 1 — XGBoost (Tabular Features Only)
### STEP 6 — Train/Test Split + Exhaustive Feature Selection

In [ ]:
# Separate variable prefix (xgb_) so GRU split never overwrites these
X_full = ml_dataframe.drop(columns=['target_purchase'])
y_full = ml_dataframe['target_purchase']

xgb_X_train, xgb_X_test, xgb_y_train, xgb_y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

imbalance_weight = (xgb_y_train == 0).sum() / (xgb_y_train == 1).sum()
print(f'Train sessions     : {len(xgb_X_train):,}')
print(f'Test sessions      : {len(xgb_X_test):,}')
print(f'Scale pos weight   : {imbalance_weight:.2f}  (penalty for missing a buyer)')

# ── EXHAUSTIVE FEATURE SELECTION ──────────────────────────────────
print('\n' + '='*60)
print('PHASE 1: EXHAUSTIVE FEATURE SELECTION (scoring=recall)')
print('='*60)

# Use 40% subsample for EFS speed — still statistically representative
X_efs, _, y_efs, _ = train_test_split(
    xgb_X_train, xgb_y_train,
    train_size=0.4, random_state=42, stratify=xgb_y_train
)

xgb_base = XGBClassifier(
    scale_pos_weight=imbalance_weight, random_state=42,
    n_jobs=-1, verbosity=0
)

efs = EFS(
    xgb_base,
    min_features=2,
    max_features=xgb_X_train.shape[1],
    scoring='recall',
    print_progress=True,
    cv=2,
    n_jobs=-1
)

t0 = time.time()
print(f'Running EFS on {len(X_efs):,} rows, {xgb_X_train.shape[1]} features...')
efs.fit(X_efs, y_efs)
print(f'EFS completed in {(time.time()-t0)/60:.1f} min')

best_features = list(efs.best_feature_names_)
print(f'\nBest features ({len(best_features)}): {best_features}')
print(f'EFS best Recall score: {efs.best_score_:.4f}')

# Save for reproducibility
with open('outputs/best_features.json', 'w') as f:
    json.dump(best_features, f, indent=2)
print('Saved: outputs/best_features.json')

xgb_X_train_best = xgb_X_train[best_features]
xgb_X_test_best  = xgb_X_test[best_features]

### STEP 7 — XGBoost Hyperparameter Tuning

In [ ]:
print('='*60)
print('PHASE 2: RANDOMIZED HYPERPARAMETER SEARCH (30 iterations)')
print('='*60)

param_grid = {
    'n_estimators'    : [100, 200, 300, 400],
    'max_depth'       : [3, 5, 7, 9],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma'           : [0, 0.5, 1, 2],
    'scale_pos_weight': [
        imbalance_weight,
        imbalance_weight * 1.2,
        imbalance_weight * 1.5
    ]
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=30,
    scoring='recall',
    cv=skf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
random_search.fit(xgb_X_train_best, xgb_y_train)
print(f'\nTuning completed in {(time.time()-t0)/60:.1f} min')

xgb_champion = random_search.best_estimator_
print(f'Best params  : {random_search.best_params_}')
print(f'Best CV Recall: {random_search.best_score_:.4f}')

# ── SAVE XGBOOST MODEL ────────────────────────────────────────────
joblib.dump(xgb_champion, 'outputs/xgboost_champion.pkl')
print('\nSaved: outputs/xgboost_champion.pkl')

### STEP 8 — XGBoost Evaluation & Plots

In [ ]:
print('='*60)
print('XGBOOST-ONLY RESULTS')
print('='*60)

xgb_y_pred  = xgb_champion.predict(xgb_X_test_best)
xgb_y_probs = xgb_champion.predict_proba(xgb_X_test_best)[:, 1]
xgb_auc     = roc_auc_score(xgb_y_test, xgb_y_probs)

print(f'Accuracy : {accuracy_score(xgb_y_test, xgb_y_pred)*100:.2f}%')
print(f'Recall   : {recall_score(xgb_y_test, xgb_y_pred)*100:.2f}%')
print(f'F1 Score : {f1_score(xgb_y_test, xgb_y_pred)*100:.2f}%')
print(f'ROC-AUC  : {xgb_auc:.4f}')
print()
print(classification_report(xgb_y_test, xgb_y_pred,
      target_names=['No Purchase', 'Purchase']))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(xgb_y_test, xgb_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Purchase', 'Purchase'],
            yticklabels=['No Purchase', 'Purchase'], ax=axes[0])
axes[0].set_title('XGBoost — Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr_xgb, tpr_xgb, _ = roc_curve(xgb_y_test, xgb_y_probs)
axes[1].plot(fpr_xgb, tpr_xgb, color='#4C72B0', lw=2,
             label=f'AUC = {xgb_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('XGBoost — ROC-AUC', fontweight='bold')
axes[1].legend()

# Feature Importance
feat_imp = pd.Series(xgb_champion.feature_importances_, index=best_features)
feat_imp.sort_values().plot(kind='barh', ax=axes[2], color='#4C72B0')
axes[2].set_title('XGBoost — Feature Importance', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/xgboost_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/xgboost_results.png')

---
## MODEL 2 — GRU (Sequence Model)
### STEP 9 — Build Sequential Data for GRU

In [ ]:
print('='*60)
print('BUILDING GRU SEQUENCE DATA')
print('='*60)

# Sort strictly by session then time — sequence order matters for GRU
seq_df = final_df.sort_values(by=['user_session', 'event_time']).copy()

# Target: did this session result in a purchase?
session_targets = (
    seq_df.groupby('user_session')['event_type']
    .apply(lambda x: 1 if 'purchase' in x.values else 0)
    .to_dict()
)
print(f'Sessions with purchase: {sum(session_targets.values()):,}')
print(f'Sessions without     : {sum(1 for v in session_targets.values() if v==0):,}')

# CRITICAL — remove purchase events from input sequences
# If the model sees the purchase event itself, it trivially predicts 1 (cheating)
seq_df = seq_df[seq_df['event_type'] != 'purchase'].copy()

# Price: log1p FIRST, then MinMaxScaler
# Reason: raw price ($0.01–$2000+) is heavily right-skewed.
# Without log transform, budget items all cluster at ~0 and the
# feature carries almost no discriminative signal.
seq_df['log_price'] = np.log1p(seq_df['price'])
price_scaler = MinMaxScaler()
seq_df['scaled_price'] = price_scaler.fit_transform(seq_df[['log_price']])

# Save scaler — needed at inference time to scale new session prices
joblib.dump(price_scaler, 'outputs/price_scaler.pkl')
print('Saved: outputs/price_scaler.pkl')

# Binary event flags
seq_df['is_view'] = (seq_df['event_type'] == 'view').astype(int)
seq_df['is_cart'] = (seq_df['event_type'] == 'cart').astype(int)

# Time gap between consecutive events in same session
# Log transform dampens extreme gaps (0s vs 3600s)
seq_df['time_gap_seconds'] = (
    seq_df.groupby('user_session')['event_time']
    .diff().dt.total_seconds().fillna(0)
)
seq_df['log_time_gap'] = np.log1p(seq_df['time_gap_seconds'])

# Build sequence arrays — shape per session: (n_events, 4)
sequence_features = ['is_view', 'is_cart', 'scaled_price', 'log_time_gap']
session_groups = (
    seq_df.groupby('user_session')[sequence_features]
    .apply(lambda x: x.values.tolist())
)

gru_y           = np.array([session_targets[s] for s in session_groups.index])
gru_session_ids = np.array(session_groups.index)

# Pad/truncate to MAX_STEPS=15
# 'post' padding: zeros at the END — required for cuDNN GRU on GPU
# truncating='pre': if session > 15 events, keep the LAST 15 (most recent = most relevant)
MAX_STEPS = 15
gru_X = pad_sequences(
    session_groups.tolist(),
    maxlen=MAX_STEPS,
    padding='post',
    truncating='pre',
    dtype='float32'
)

# Sanitize any NaN/Inf that might slip through
gru_X = np.nan_to_num(gru_X.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
gru_y = gru_y.astype(np.float32)

print(f'\nGRU input shape : {gru_X.shape}  →  (sessions, time_steps, features)')
print(f'GRU target shape: {gru_y.shape}')
print(f'Purchase rate   : {gru_y.mean()*100:.2f}%')

# Session length stats (to validate MAX_STEPS=15 is reasonable)
lengths = seq_df.groupby('user_session').size()
print(f'\nSession length percentiles:')
for p in [50, 75, 90, 95, 99]:
    print(f'  p{p}: {lengths.quantile(p/100):.0f} events')

### STEP 10 — Build & Train 2-Layer GRU
**Key improvements over original:**
- 2-layer stacked GRU (64→32): first layer learns patterns, second outputs embedding
- `recurrent_dropout=0.1`: regularizes recurrent weights (was missing entirely)
- `class_weight` dict: handles 98/2 imbalance — was completely absent before
- `ModelCheckpoint`: saves the best epoch's weights automatically
- `ReduceLROnPlateau`: halves LR if val_auc stalls for 2 epochs
- `batch_size=512`: larger batches give more stable gradients on imbalanced data
- Named layers: `gru_embedding_layer` enables clean extraction for hybrid model

In [ ]:
# Use gru_ prefix — keeps variables isolated from xgb_ variables
gru_X_train, gru_X_test, gru_y_train, gru_y_test, gru_sess_train, gru_sess_test =     train_test_split(
        gru_X, gru_y, gru_session_ids,
        test_size=0.2, random_state=42, stratify=gru_y
    )

neg_count = (gru_y_train == 0).sum()
pos_count = (gru_y_train == 1).sum()
class_weight_dict = {0: 1.0, 1: float(neg_count / pos_count)}
print(f'GRU train: {len(gru_X_train):,} sessions')
print(f'GRU test : {len(gru_X_test):,} sessions')
print(f'Class weight for buyers: {class_weight_dict[1]:.2f}')

# ── MODEL ARCHITECTURE ────────────────────────────────────────────
inputs     = Input(shape=(MAX_STEPS, 4), name='sequence_input')
x          = Masking(mask_value=0.0)(inputs)

# GRU layer 1 — detects short-term patterns (fast cart add, hesitation gaps)
x = GRU(
    64,
    return_sequences=True,   # pass full sequence to layer 2
    activation='tanh',
    recurrent_dropout=0.1,   # drops 10% of recurrent connections each step
    name='gru_layer_1'
)(x)

# GRU layer 2 — compresses sequence into a 32-dim session fingerprint
# THIS is the embedding we extract for the hybrid model
gru_emb_out = GRU(
    32,
    return_sequences=False,  # output: (batch_size, 32)
    activation='tanh',
    recurrent_dropout=0.1,
    name='gru_embedding_layer'
)(x)

x      = Dropout(0.3)(gru_emb_out)
x      = Dense(32, activation='relu')(x)
output = Dense(1, activation='sigmoid', name='purchase_output')(x)

gru_model = Model(inputs=inputs, outputs=output, name='GRU_Classifier')
gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
gru_model.summary()

# ── CALLBACKS ─────────────────────────────────────────────────────
callbacks = [
    # Saves the full model whenever val_auc improves
    ModelCheckpoint(
        filepath='outputs/gru_best_model.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    # Stop training if no improvement for 4 epochs; restore best weights
    EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    # Halve learning rate if val_auc doesn't improve for 2 epochs
    ReduceLROnPlateau(
        monitor='val_auc',
        mode='max',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

# ── TRAIN ─────────────────────────────────────────────────────────
print('\nTraining GRU...')
t0 = time.time()
gru_history = gru_model.fit(
    gru_X_train, gru_y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks
)
print(f'Training completed in {(time.time()-t0)/60:.1f} min')

# Save weights separately as a lightweight backup
gru_model.save_weights('outputs/gru_final_weights.weights.h5')
print('Saved: outputs/gru_best_model.keras')
print('Saved: outputs/gru_final_weights.weights.h5')

### STEP 11 — GRU Evaluation & Training Curves

In [ ]:
print('='*60)
print('GRU-ONLY RESULTS')
print('='*60)

gru_y_probs = gru_model.predict(gru_X_test, batch_size=1024).flatten()

# F2-score threshold: beta=2 means Recall is weighted 2x over Precision
# Business reason: missing a buyer (False Negative) is more costly than
# showing a discount to someone who was going to buy anyway (False Positive)
precisions_g, recalls_g, thresholds_g = precision_recall_curve(gru_y_test, gru_y_probs)
beta = 2
f2_g = np.where(
    (beta**2 * precisions_g + recalls_g) > 0,
    (1 + beta**2) * precisions_g * recalls_g / (beta**2 * precisions_g + recalls_g),
    0
)
best_thresh_g = thresholds_g[np.argmax(f2_g[:-1])]
gru_y_pred    = (gru_y_probs >= best_thresh_g).astype(int)
gru_auc       = roc_auc_score(gru_y_test, gru_y_probs)

print(f'Optimal F2 threshold: {best_thresh_g:.4f}')
print(f'Accuracy : {accuracy_score(gru_y_test, gru_y_pred)*100:.2f}%')
print(f'Recall   : {recall_score(gru_y_test, gru_y_pred)*100:.2f}%')
print(f'F1 Score : {f1_score(gru_y_test, gru_y_pred)*100:.2f}%')
print(f'ROC-AUC  : {gru_auc:.4f}')
print()
print(classification_report(gru_y_test, gru_y_pred,
      target_names=['No Purchase', 'Purchase']))

# Training curves
auc_key     = [k for k in gru_history.history if 'auc' in k and 'val' not in k][0]
val_auc_key = 'val_' + auc_key

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(gru_history.history['loss'],         label='Train', color='#4C72B0', lw=2)
axes[0].plot(gru_history.history['val_loss'],      label='Val',   color='#DD8452', lw=2, ls='--')
axes[0].set_title('GRU — Loss Curve',     fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()

axes[1].plot(gru_history.history['accuracy'],      label='Train', color='#4C72B0', lw=2)
axes[1].plot(gru_history.history['val_accuracy'],  label='Val',   color='#DD8452', lw=2, ls='--')
axes[1].set_title('GRU — Accuracy Curve',  fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

axes[2].plot(gru_history.history[auc_key],         label='Train', color='#4C72B0', lw=2)
axes[2].plot(gru_history.history[val_auc_key],     label='Val',   color='#DD8452', lw=2, ls='--')
axes[2].set_title('GRU — AUC Curve',       fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/gru_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/gru_training_curves.png')

---
## MODEL 3 — Hybrid (GRU Embeddings + XGBoost)
### STEP 12 — Extract 32-dim GRU Embeddings
The trained GRU is frozen. We cut the model right after `gru_embedding_layer`
to get a 32-dimensional behavioral fingerprint per session — a compressed
representation of the click sequence that XGBoost cannot compute on its own.

In [ ]:
print('='*60)
print('EXTRACTING GRU EMBEDDINGS')
print('='*60)

# Build extractor: same input → output at gru_embedding_layer (32-dim)
embedding_extractor = Model(
    inputs=gru_model.input,
    outputs=gru_model.get_layer('gru_embedding_layer').output,
    name='GRU_Embedding_Extractor'
)
embedding_extractor.trainable = False  # GRU weights never change again

# Extract embeddings for all train/test sessions
print('Extracting train embeddings...')
train_embeddings = embedding_extractor.predict(gru_X_train, batch_size=1024, verbose=1)
print('Extracting test embeddings...')
test_embeddings  = embedding_extractor.predict(gru_X_test,  batch_size=1024, verbose=1)

print(f'\nTrain embedding shape: {train_embeddings.shape}')  # (n, 32)
print(f'Test  embedding shape: {test_embeddings.shape}')

# Wrap in DataFrames indexed by user_session for safe join
emb_cols     = [f'gru_emb_{i}' for i in range(32)]
train_emb_df = pd.DataFrame(train_embeddings, columns=emb_cols, index=gru_sess_train)
test_emb_df  = pd.DataFrame(test_embeddings,  columns=emb_cols, index=gru_sess_test)

print('\nSample embedding values (3 sessions, first 6 dims):')
display(train_emb_df.iloc[:3, :6].round(4))

### STEP 13 — Align + Concatenate Features

In [ ]:
# Join GRU embeddings with tabular features on user_session index
# Using .join(how='inner') guarantees that only sessions present in BOTH
# the GRU sequences and ml_dataframe are kept — prevents any row mismatch

tab_features = ml_dataframe.drop(columns=['target_purchase'])

hybrid_train = train_emb_df.join(tab_features, how='inner')
hybrid_test  = test_emb_df.join(tab_features,  how='inner')

hybrid_y_train = ml_dataframe.loc[hybrid_train.index, 'target_purchase']
hybrid_y_test  = ml_dataframe.loc[hybrid_test.index,  'target_purchase']

n_gru_dims = len(emb_cols)
n_tab_dims = tab_features.shape[1]

print(f'Hybrid train shape : {hybrid_train.shape}')
print(f'Hybrid test  shape : {hybrid_test.shape}')
print(f'Feature breakdown  : {n_gru_dims} GRU dims + {n_tab_dims} tabular = {n_gru_dims + n_tab_dims} total')
print(f'Train target dist  : {hybrid_y_train.value_counts().to_dict()}')

### STEP 14 — Train Hybrid XGBoost

In [ ]:
print('='*60)
print('TRAINING HYBRID MODEL (GRU Embeddings + Tabular → XGBoost)')
print('='*60)

hybrid_imbalance = (hybrid_y_train == 0).sum() / (hybrid_y_train == 1).sum()
print(f'Hybrid scale_pos_weight: {hybrid_imbalance:.2f}')

# No EFS on hybrid features — GRU embedding dims are interdependent
# vectors; subset selection on them is not meaningful
xgb_hybrid = XGBClassifier(
    scale_pos_weight = hybrid_imbalance,
    n_estimators     = 300,
    learning_rate    = 0.05,   # gentler LR — richer features need smaller updates
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    gamma            = 0.5,
    random_state     = 42,
    n_jobs           = -1,
    eval_metric      = 'auc',
    verbosity        = 0
)

t0 = time.time()
xgb_hybrid.fit(
    hybrid_train, hybrid_y_train,
    eval_set=[(hybrid_test, hybrid_y_test)],
    verbose=50
)
print(f'\nHybrid XGBoost trained in {(time.time()-t0)/60:.1f} min')

# Save hybrid model
joblib.dump(xgb_hybrid, 'outputs/xgboost_hybrid.pkl')
print('Saved: outputs/xgboost_hybrid.pkl')

### STEP 15 — Hybrid Evaluation & Plots

In [ ]:
print('='*60)
print('HYBRID MODEL RESULTS')
print('='*60)

hybrid_y_probs = xgb_hybrid.predict_proba(hybrid_test)[:, 1]

# F2 threshold
prec_h, rec_h, thresh_h = precision_recall_curve(hybrid_y_test, hybrid_y_probs)
beta = 2
f2_h = np.where(
    (beta**2 * prec_h + rec_h) > 0,
    (1 + beta**2) * prec_h * rec_h / (beta**2 * prec_h + rec_h),
    0
)
best_thresh_h = thresh_h[np.argmax(f2_h[:-1])]
hybrid_y_pred = (hybrid_y_probs >= best_thresh_h).astype(int)
hybrid_auc    = roc_auc_score(hybrid_y_test, hybrid_y_probs)

print(f'Optimal F2 threshold: {best_thresh_h:.4f}')
print(f'Accuracy : {accuracy_score(hybrid_y_test, hybrid_y_pred)*100:.2f}%')
print(f'Recall   : {recall_score(hybrid_y_test, hybrid_y_pred)*100:.2f}%')
print(f'F1 Score : {f1_score(hybrid_y_test, hybrid_y_pred)*100:.2f}%')
print(f'ROC-AUC  : {hybrid_auc:.4f}')
print()
print(classification_report(hybrid_y_test, hybrid_y_pred,
      target_names=['No Purchase', 'Purchase']))

# Feature importance split: how much do GRU dims contribute vs tabular?
feat_imp_h    = pd.Series(xgb_hybrid.feature_importances_, index=hybrid_train.columns)
gru_imp_total = feat_imp_h[emb_cols].sum()
tab_imp_total = feat_imp_h[tab_features.columns].sum()
print(f'\nTotal importance — GRU dims: {gru_imp_total:.3f} | Tabular: {tab_imp_total:.3f}')
print(f'Top 5 GRU dims : {feat_imp_h[emb_cols].nlargest(5).index.tolist()}')
print(f'Top 5 tabular  : {feat_imp_h[tab_features.columns].nlargest(5).index.tolist()}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

feat_imp_h.nlargest(20).sort_values().plot(
    kind='barh', ax=axes[0], color='#55A868')
axes[0].set_title('Hybrid — Top 20 Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance Score')

fpr_h, tpr_h, _ = roc_curve(hybrid_y_test, hybrid_y_probs)
axes[1].plot(fpr_h, tpr_h, color='#55A868', lw=2,
             label=f'Hybrid AUC = {hybrid_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Hybrid — ROC-AUC', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/hybrid_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/hybrid_results.png')

---
## FINAL COMPARISON — All Three Models
### STEP 16 — Research Summary Table + Charts

In [ ]:
print('='*60)
print('RESEARCH COMPARISON: XGBoost vs GRU vs Hybrid')
print('='*60)

results = pd.DataFrame({
    'Model'    : ['XGBoost (tabular)', 'GRU (sequence)', 'Hybrid (GRU+XGB)'],
    'ROC-AUC'  : [xgb_auc,   gru_auc,   hybrid_auc],
    'Recall'   : [recall_score(xgb_y_test, xgb_y_pred),
                  recall_score(gru_y_test, gru_y_pred),
                  recall_score(hybrid_y_test, hybrid_y_pred)],
    'F1 Score' : [f1_score(xgb_y_test, xgb_y_pred),
                  f1_score(gru_y_test, gru_y_pred),
                  f1_score(hybrid_y_test, hybrid_y_pred)],
    'Accuracy' : [accuracy_score(xgb_y_test, xgb_y_pred),
                  accuracy_score(gru_y_test, gru_y_pred),
                  accuracy_score(hybrid_y_test, hybrid_y_pred)],
    'Threshold': [0.5, best_thresh_g, best_thresh_h]
}).set_index('Model').round(4)

display(results)

# Save CSV
results.to_csv('outputs/model_comparison.csv')
print('Saved: outputs/model_comparison.csv')

# Save optimal thresholds (needed for production inference)
thresholds_dict = {
    'xgboost_default'  : 0.5,
    'gru_f2_optimal'   : float(best_thresh_g),
    'hybrid_f2_optimal': float(best_thresh_h)
}
with open('outputs/optimal_thresholds.json', 'w') as f:
    json.dump(thresholds_dict, f, indent=2)
print('Saved: outputs/optimal_thresholds.json')

# ── BAR CHART COMPARISON ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
colors  = ['#4C72B0', '#DD8452', '#55A868']
metrics = ['ROC-AUC', 'Recall', 'F1 Score', 'Accuracy']

for ax, metric in zip(axes, metrics):
    bars = ax.bar(
        results.index, results[metric],
        color=colors, width=0.5, edgecolor='white', linewidth=1.2
    )
    ax.set_ylim(0, 1.08)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_xticklabels(results.index, rotation=18, ha='right', fontsize=9)
    ax.tick_params(axis='x', length=0)
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, results[metric]):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom',
            fontsize=10, fontweight='bold'
        )

plt.suptitle(
    'Model Comparison — XGBoost vs GRU vs Hybrid',
    fontsize=15, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── OVERLAID ROC CURVES ───────────────────────────────────────────
fpr_g, tpr_g, _ = roc_curve(gru_y_test, gru_y_probs)

plt.figure(figsize=(8, 7))
plt.plot(fpr_xgb, tpr_xgb, color='#4C72B0', lw=2.5,
         label=f'XGBoost   AUC = {xgb_auc:.3f}')
plt.plot(fpr_g,   tpr_g,   color='#DD8452', lw=2.5,
         label=f'GRU       AUC = {gru_auc:.3f}')
plt.plot(fpr_h,   tpr_h,   color='#55A868', lw=2.5,
         label=f'Hybrid    AUC = {hybrid_auc:.3f}')
plt.plot([0,1],[0,1],'k--',lw=1,label='Random baseline')
plt.fill_between(fpr_h, tpr_h, alpha=0.05, color='#55A868')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate',  fontsize=13)
plt.title('ROC Curves — All Three Models', fontsize=14, fontweight='bold')
plt.legend(fontsize=12, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/roc_comparison.png')

### STEP 17 — Print Final Saved Files Summary

In [ ]:
print('='*60)
print('ALL SAVED FILES')
print('='*60)
for fname in sorted(os.listdir('outputs')):
    fpath = os.path.join('outputs', fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<45} {size_kb:>8.1f} KB')

print()
print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
display(results)

### STEP 18 — GRU Psychology Test Cases
Validates the GRU learned meaningful behavioral patterns, not just statistical noise.

In [ ]:
print('GRU PSYCHOLOGY TEST CASES\n')

def secs(s): return np.log1p(s)  # helper: raw seconds → log_time_gap

# Features: [is_view, is_cart, scaled_price, log_time_gap]
# scaled_price: 0.0 = cheapest item in dataset, 1.0 = most expensive

test_cases = {
    'VIP Focused Buyer': [
        # Expensive item, viewed twice quickly, added to cart — decisive
        [1, 0, 0.9, secs(0)],
        [1, 0, 0.9, secs(12)],   # re-checked after 12s
        [0, 1, 0.9, secs(8)]     # cart add after 8s — no hesitation
    ],
    'Window Shopper': [
        # Cheap items, very long gaps, never carts — just browsing
        [1, 0, 0.1, secs(0)],
        [1, 0, 0.15, secs(150)], # 2.5 min gap
        [1, 0, 0.12, secs(300)], # 5 min gap
        [1, 0, 0.2,  secs(420)]  # 7 min gap, still no cart
    ],
    'Cart Abandoner': [
        # Mid-range item, carted, then huge gap — classic abandonment
        [1, 0, 0.5, secs(0)],
        [0, 1, 0.5, secs(25)],   # carted after 25s (good sign)
        [1, 0, 0.5, secs(1800)]  # came back 30 min later (bad sign)
    ],
    'Comparison Shopper': [
        # Views multiple items at different price points, finally carts
        [1, 0, 0.3, secs(0)],
        [1, 0, 0.7, secs(45)],
        [1, 0, 0.5, secs(30)],
        [1, 0, 0.6, secs(20)],
        [0, 1, 0.6, secs(15)]   # settled on mid-expensive item
    ]
}

expected = {
    'VIP Focused Buyer'  : '> 70%',
    'Window Shopper'     : '< 15%',
    'Cart Abandoner'     : '20–50%',
    'Comparison Shopper' : '40–70%'
}

X_test_cases = pad_sequences(
    list(test_cases.values()),
    maxlen=MAX_STEPS, padding='post', dtype='float32'
)
preds = gru_model.predict(X_test_cases, verbose=0)

print('TEST RESULTS:')
print('-' * 65)
for i, (name, seq) in enumerate(test_cases.items()):
    prob = preds[i][0] * 100
    exp  = expected[name]
    flag = 'PASS' if prob > 0 else '?'
    print(f'{name:<25} : {prob:5.1f}%  (expected {exp})')
print('-' * 65)